In [1]:
print("Hola mundo")

Hola mundo


***Conectando a Drive y Accediendo al Archivo índice comprimido (´archive_listing.txt´)***

In [2]:
from google.colab import drive
import os

# 1. Montar Google Drive en la sesión
drive.mount('/content/drive')

# 2. Definir la ruta donde subiste el archivo en Drive
# Cambia 'data/archive_listing.txt' si lo pusiste en otra carpeta
TXT_PATH = '/content/drive/MyDrive/BiSID5k/archive_listing.txt'

# 3. Comprobar que Colab detecta el archivo
if os.path.exists(TXT_PATH):
    print("✅ ¡Perfecto! El archivo archive_listing.txt fue encontrado en Google Drive.")
else:
    print("❌ Archivo no encontrado. Revisa la ruta dentro de tu Google Drive.")

Mounted at /content/drive
✅ ¡Perfecto! El archivo archive_listing.txt fue encontrado en Google Drive.


# **Inspección de la estructura raíz y total de entradas**

In [3]:
# Cargar todas las líneas del archivo
with open(TXT_PATH, 'r', encoding='utf-8') as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]

print(f"Total de registros listados en el dataset: {len(lines):,}\n")

# Mostrar las primeras 30 líneas para entender el directorio raíz y las carpetas iniciales
print("--- PRIMERAS 30 LÍNEAS DEL ARCHIVO (Estructura Raíz) ---")
for line in lines[:30]:
    print(line)

Total de registros listados en el dataset: 15,054

--- PRIMERAS 30 LÍNEAS DEL ARCHIVO (Estructura Raíz) ---
### BiSID-5k.tar.gz ###
BiSID-5k/
BiSID-5k/bg_images/
BiSID-5k/bg_images/5/
BiSID-5k/bg_images/5/._RGB_Raw.jpg
BiSID-5k/bg_images/5/RGB_Raw.jpg
BiSID-5k/bg_images/5/HS_Raw.zip
BiSID-5k/bg_images/4/
BiSID-5k/bg_images/4/._RGB_Raw.jpg
BiSID-5k/bg_images/4/RGB_Raw.jpg
BiSID-5k/bg_images/4/HS_Raw.zip
BiSID-5k/bg_images/6/
BiSID-5k/bg_images/6/._RGB_Raw.jpg
BiSID-5k/bg_images/6/RGB_Raw.jpg
BiSID-5k/bg_images/6/HS_Raw.zip
BiSID-5k/bg_images/8/
BiSID-5k/bg_images/8/._RGB_Raw.jpg
BiSID-5k/bg_images/8/RGB_Raw.jpg
BiSID-5k/bg_images/8/HS_Raw.zip
BiSID-5k/bg_images/2/
BiSID-5k/bg_images/2/._RGB_Raw.jpg
BiSID-5k/bg_images/2/RGB_Raw.jpg
BiSID-5k/bg_images/2/HS_Raw.zip
BiSID-5k/bg_images/3/
BiSID-5k/bg_images/3/._RGB_Raw.jpg
BiSID-5k/bg_images/3/RGB_Raw.jpg
BiSID-5k/bg_images/3/HS_Raw.zip
BiSID-5k/bg_images/9/
BiSID-5k/bg_images/9/._RGB_Raw.jpg
BiSID-5k/bg_images/9/RGB_Raw.jpg


# **Creando el inventario filtrado de imágenes ´RGB_Raw.jpg´**

In [5]:
import pandas as pd

# 1. Filtrar únicamente las imágenes RGB principales
rgb_paths = [line for line in lines if line.endswith('RGB_Raw.jpg')]

print(f"Total de imágenes 'RGB_Raw.jpg' encontradas: {len(rgb_paths)}")

# 2. Descomponer las rutas para estructurar el inventario
data = []
for path in rgb_paths:
    parts = path.split('/')
    # Ejemplo típico: BiSID-5k/bg_images/5/RGB_Raw.jpg
    data.append({
        'ruta_completa': path,
        'directorio_raiz': parts[0] if len(parts) > 0 else None,
        'categoria_padre': parts[1] if len(parts) > 1 else None,
        'id_muestra': parts[2] if len(parts) > 2 else None,
        'nombre_archivo': parts[-1]
    })

df_rgb = pd.DataFrame(data)

# 3. Mostrar la tabla inicial (Primeros 15 registros)
print("\n--- INVENTARIO INICIAL DE IMÁGENES RGB (df_rgb) ---")
display(df_rgb.head(15))

# 4. Ver cómo se distribuyen las carpetas o categorías principales
print("\n--- DISTRIBUCIÓN POR CATEGORÍA PADRE ---")
print(df_rgb['categoria_padre'].value_counts())

Total de imágenes 'RGB_Raw.jpg' encontradas: 5020

--- INVENTARIO INICIAL DE IMÁGENES RGB (df_rgb) ---


,ruta_completa,directorio_raiz,categoria_padre,id_muestra,nombre_archivo
0,BiSID-5k/bg_images/5/._RGB_Raw.jpg,BiSID-5k,bg_images,5,._RGB_Raw.jpg
1,BiSID-5k/bg_images/5/RGB_Raw.jpg,BiSID-5k,bg_images,5,RGB_Raw.jpg
2,BiSID-5k/bg_images/4/._RGB_Raw.jpg,BiSID-5k,bg_images,4,._RGB_Raw.jpg
3,BiSID-5k/bg_images/4/RGB_Raw.jpg,BiSID-5k,bg_images,4,RGB_Raw.jpg
4,BiSID-5k/bg_images/6/._RGB_Raw.jpg,BiSID-5k,bg_images,6,._RGB_Raw.jpg
5,BiSID-5k/bg_images/6/RGB_Raw.jpg,BiSID-5k,bg_images,6,RGB_Raw.jpg
6,BiSID-5k/bg_images/8/._RGB_Raw.jpg,BiSID-5k,bg_images,8,._RGB_Raw.jpg
7,BiSID-5k/bg_images/8/RGB_Raw.jpg,BiSID-5k,bg_images,8,RGB_Raw.jpg
8,BiSID-5k/bg_images/2/._RGB_Raw.jpg,BiSID-5k,bg_images,2,._RGB_Raw.jpg
9,BiSID-5k/bg_images/2/RGB_Raw.jpg,BiSID-5k,bg_images,2,RGB_Raw.jpg



--- DISTRIBUCIÓN POR CATEGORÍA PADRE ---
categoria_padre
images       5000
bg_images      20
Name: count, dtype: int64


# **Análisis de formatos adicionales y archivos ocultos**

In [6]:
# 1. Conocer todos los tipos de extensiones de archivo presentes en el dataset
extensiones = pd.Series([line.split('.')[-1] for line in lines if '.' in line]).value_counts()
print("--- TIPOS DE ARCHIVOS / EXTENSIONES EN EL DATASET ---")
print(extensiones)

# 2. Identificar archivos de sistema u ocultos (archivos de macOS/Linux que empiezan por ._)
archivos_ocultos = [line for line in lines if '/._' in line]
print(f"\nTotal de archivos ocultos de metadatos (ej. ._RGB_Raw.jpg): {len(archivos_ocultos)}")

# 3. Conocer la cantidad total de muestras únicas registradas
muestras_unicas = df_rgb['id_muestra'].dropna().unique()
print(f"\nTotal de IDs de muestras únicas identificadas: {len(muestras_unicas)}")

--- TIPOS DE ARCHIVOS / EXTENSIONES EN EL DATASET ---
jpg       5020
zip       5010
gz ###       1
Name: count, dtype: int64

Total de archivos ocultos de metadatos (ej. ._RGB_Raw.jpg): 10

Total de IDs de muestras únicas identificadas: 20


## Conclusiones de la Inspección de la Estructura Real

Tras analizar el archivo de índice `archive_listing.txt`, se determinó la estructura interna real del dataset BiSID-5k sin necesidad de descomprimir el paquete completo de 1 TB:

### 1. Hallazgos de la Estructura
* **Directorio raíz:** `BiSID-5k/`
* **Carpetas principales (`categoria_padre`):**
  * `images/`: Contiene las **5,000 imágenes sintéticas principales** del dataset.
  * `bg_images/`: Contiene **20 imágenes de fondo** utilizadas para la generación sintética.
* **Ubicación de imágenes RGB:** Se encuentran en la ruta `BiSID-5k/<categoria_padre>/<id_muestra>/RGB_Raw.jpg`.

### 2. Inventario de Archivos
* **Total de imágenes `RGB_Raw.jpg` válidas:** 5,020 (5,000 principales + 20 de fondo).
* **Archivos hiperespectrales (`HS_Raw.zip`):** 5,010 archivos compressed zip.
* **Archivos basura/metadatos:** Se identificaron **10 archivos ocultos de sistema** (`._RGB_Raw.jpg`), los cuales deben excluirse mediante filtrado explícito en la preparación de datos.

### 3. Siguiente Paso
Con las rutas estructuradas en el DataFrame `df_rgb`, tenemos la tabla inicial lista para proceder con la selección de muestras, la creación del archivo de divisiones (train/val/test) y la descarga selectiva de únicamente las imágenes RGB.